# Data Cleaning Pipeline

In [1]:
import pandas as pd
import numpy as np
import os


## 1 · Load Raw Data

In [2]:
customers   = pd.read_csv("../data/raw/customers.csv")
events      = pd.read_csv("../data/raw/events.csv")
orders      = pd.read_csv("../data/raw/orders.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
products    = pd.read_csv("../data/raw/products.csv")
reviews     = pd.read_csv("../data/raw/reviews.csv")
sessions    = pd.read_csv("../data/raw/sessions.csv")

print("Datasets loaded successfully")


Datasets loaded successfully


## 2 · Remove Duplicates

In [3]:
# Drop fully-duplicate rows from all tables
for df in [customers, events, orders, order_items, products, reviews, sessions]:
    df.drop_duplicates(inplace=True)

# Deduplicate on business keys — keep last record per ID
orders    = orders.sort_values("order_time", na_position="first")                  .drop_duplicates(subset=["order_id"],    keep="last").reset_index(drop=True)
customers = customers.drop_duplicates(subset=["customer_id"], keep="last").reset_index(drop=True)
products  = products.drop_duplicates(subset=["product_id"],  keep="last").reset_index(drop=True)

print("Duplicates removed")


Duplicates removed


## 3 · Handle Missing Values

In [4]:
# Prices → fill with median
order_items["unit_price_usd"] = order_items["unit_price_usd"]    .fillna(order_items["unit_price_usd"].median())

# Category → replace with "Unknown"
products["category"] = products["category"].fillna("Unknown")

# Product name → replace with "Unknown Product"
if "name" in products.columns:
    products["name"] = products["name"].fillna("Unknown Product")

# Orders with no customer_id are orphans — drop them
orders = orders.dropna(subset=["customer_id"])

# Review ratings → fill with median
reviews["rating"] = reviews["rating"].fillna(reviews["rating"].median())

print("Missing values handled")


Missing values handled


## 4 · Normalize Product Names & IDs

In [5]:
# Product names → lowercase and strip whitespace
products["name"] = products["name"].astype(str).str.lower().str.strip()

# All ID columns → consistent string type to prevent join mismatches
for df, cols in [
    (products,    ["product_id"]),
    (order_items, ["product_id", "order_id"]),
    (orders,      ["order_id",   "customer_id"]),
    (customers,   ["customer_id"]),
    (reviews,     ["customer_id", "product_id"]),
]:
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

print("Names and IDs normalized")


Names and IDs normalized


## 5 · Validate Timestamps

In [6]:
now = pd.Timestamp.now()

# Orders — parse, drop invalid, drop future dates
orders["order_time"] = pd.to_datetime(orders["order_time"], errors="coerce")
orders = orders.dropna(subset=["order_time"])
orders = orders[orders["order_time"] <= now]

# Events — try common timestamp column names
for col in ["event_time", "timestamp", "created_at"]:
    if col in events.columns:
        events[col] = pd.to_datetime(events[col], errors="coerce")
        events = events.dropna(subset=[col])
        events = events[events[col] <= now]
        break

# Sessions — try common timestamp column names
for col in ["session_start", "started_at", "timestamp"]:
    if col in sessions.columns:
        sessions[col] = pd.to_datetime(sessions[col], errors="coerce")
        sessions = sessions.dropna(subset=[col])
        sessions = sessions[sessions[col] <= now]
        break

print("Timestamps validated")


Timestamps validated


## 6 · Validate Numeric Values

In [7]:
# Remove rows with non-positive price or quantity
order_items = order_items[order_items["unit_price_usd"] > 0]
order_items = order_items[order_items["quantity"]       > 0]

# Remove out-of-range ratings
reviews = reviews[reviews["rating"].between(1, 5)]

print("Numeric values validated")


Numeric values validated


## 7 · Currency Consistency

In [8]:
# Keep only USD orders if a currency column exists
if "currency" in orders.columns:
    orders = orders[orders["currency"].str.upper() == "USD"].reset_index(drop=True)
    print("Kept USD orders only")
else:
    print("No currency column — skipping")


No currency column — skipping


## 8 · Final Type Enforcement

In [9]:
order_items["quantity"]       = order_items["quantity"].astype(int)
order_items["unit_price_usd"] = order_items["unit_price_usd"].astype(float)
order_items["revenue_usd"]    = order_items["quantity"] * order_items["unit_price_usd"]

print("Types enforced")


Types enforced


## 9 · Summary

In [10]:
print("=" * 45)
print("CLEANED DATASET SUMMARY")
print("=" * 45)
for name, df in [("customers", customers), ("events", events), ("orders", orders),
                 ("order_items", order_items), ("products", products),
                 ("reviews", reviews), ("sessions", sessions)]:
    print(f"  {name:12s}: {len(df):>7,} rows  |  {df.isnull().sum().sum()} nulls remaining")
print("=" * 45)


CLEANED DATASET SUMMARY
  customers   :  20,000 rows  |  0 nulls remaining
  events      : 760,958 rows  |  3594504 nulls remaining
  orders      :  33,580 rows  |  0 nulls remaining
  order_items :  59,090 rows  |  0 nulls remaining
  products    :   1,197 rows  |  0 nulls remaining
  reviews     :  10,780 rows  |  0 nulls remaining
  sessions    : 120,000 rows  |  0 nulls remaining


## 10 · Save Cleaned Datasets

In [11]:
os.makedirs("../data/processed", exist_ok=True)

customers.to_csv("../data/processed/customers_clean.csv",     index=False)
events.to_csv("../data/processed/events_clean.csv",           index=False)
orders.to_csv("../data/processed/orders_clean.csv",           index=False)
order_items.to_csv("../data/processed/order_items_clean.csv", index=False)
products.to_csv("../data/processed/products_clean.csv",       index=False)
reviews.to_csv("../data/processed/reviews_clean.csv",         index=False)
sessions.to_csv("../data/processed/sessions_clean.csv",       index=False)

print("All cleaned datasets saved to ../data/processed/")


All cleaned datasets saved to ../data/processed/
